In [1]:
import re
import pandas as pd

# ── 1. Disease keyword dictionary ──────────────────────────────────────────────
diseases = {
    "pneumonia":    ["pneumonia", "infection", "infectious", "aspiration"],
    "edema":        ["edema", "pulmonary edema", "oedema", "vascular congestion",
                     "pulmonary vascular congestion", "interstitial edema",
                     "interstitial abnormality", "vascular prominence"],
    "effusion":     ["effusion", "pleural effusion", "pleural effusions",
                     "layering fluid", "loculated fluid"],
    "pneumothorax": ["pneumothorax"],
    "cardiomegaly": ["cardiomegaly", "enlarged heart", "heart is enlarged",
                     "cardiac silhouette is enlarged", "heart size is enlarged",
                     "mild cardiomegaly", "moderate cardiomegaly",
                     "cardiac silhouette is bigger", "mildly enlarged",
                     "borderline enlarged"],
    "atelectasis":  ["atelectasis", "volume loss", "subsegmental atelectasis",
                     "basilar atelectasis", "bibasilar atelectasis",
                     "linear atelectasis", "compressive atelectasis",
                     "discoid atelectasis"],
    "consolidation":["consolidation", "airspace disease", "airspace opacity",
                     "airspace opacities", "lobar consolidation"],
    "opacity":      ["opacity", "opacification", "infiltrate", "infiltrates",
                     "density", "haziness"],
}

# ── 2. Negation & uncertainty phrase lists ──────────────────────────────────────
NEGATION_PHRASES = [
    "no evidence of", "no signs of", "no definite", "no focal", "no acute",
    "no large", "no convincing", "no overt", "no residual",
    "without", "free of", "negative for", "ruled out", "rule out",
    "not seen", "resolved", "clear of", "absent", "no ",
]

UNCERTAINTY_PHRASES = [
    "cannot exclude", "difficult to exclude", "hard to exclude",
    "cannot rule out", "not excluded", "could not be excluded",
    "possible", "probable", "may represent", "may reflect", "may be",
    "might", "suggest", "suggests", "suggesting",
    "concerning for", "suspicious for", "worrisome",
    "could represent", "could reflect", "likely", "appears to be",
    "consistent with", "versus", " vs ", "question of",
    "would have to be considered", "superimposed",
]

# ── 3. Core sentence-level classifier ──────────────────────────────────────────
def _classify_in_sentence(sentence: str, keywords: list) -> int:
    """Return 1 (positive), -1 (uncertain), or 0 (absent) for one sentence."""
    s = sentence.lower()

    for kw in keywords:
        if kw not in s:
            continue

        # Build a small window around the keyword occurrence
        idx = s.index(kw)
        window = s[max(0, idx - 80): idx + len(kw) + 80]

        # Check negation — must appear BEFORE the keyword in the window
        before = s[max(0, idx - 80): idx]
        negated = any(neg in before for neg in NEGATION_PHRASES)
        if negated:
            continue  # treat as absent

        # Check uncertainty anywhere in window
        uncertain = any(unc in window for unc in UNCERTAINTY_PHRASES)
        if uncertain:
            return -1  # uncertain positive

        return 1  # clear positive

    return 0  # keyword not found (or all negated)


# ── 4. Multi-sentence / multi-report aggregator ────────────────────────────────
def _split_sentences(text: str) -> list:
    """Rough sentence splitter suitable for radiology text."""
    # Split on period-space, numbered lists, or newlines
    parts = re.split(r'(?<=[.!?])\s+|\n+|\.\s*\d+\.', text)
    return [p.strip() for p in parts if p.strip()]


def classify_report(text: str) -> dict:
    """Label a single report string → {disease: 0/1/-1}."""
    labels = {d: 0 for d in diseases}
    sentences = _split_sentences(text)

    for sentence in sentences:
        for disease, keywords in diseases.items():
            result = _classify_in_sentence(sentence, keywords)
            # Positive beats uncertain beats absent
            if result == 1:
                labels[disease] = 1
            elif result == -1 and labels[disease] == 0:
                labels[disease] = -1

    return labels


def combine_report_list(texts) -> dict:
    """
    Accept either a single string or a list of strings (multiple views /
    time-points). Aggregate across all of them.
    """
    if isinstance(texts, str):
        texts = [texts]
    if not isinstance(texts, list):
        texts = list(texts)

    combined = {d: 0 for d in diseases}
    for text in texts:
        if not isinstance(text, str) or not text.strip():
            continue
        report_labels = classify_report(text)
        for d in diseases:
            if report_labels[d] == 1:
                combined[d] = 1
            elif report_labels[d] == -1 and combined[d] == 0:
                combined[d] = -1

    return combined


# ── 5. Normal-study helper ─────────────────────────────────────────────────────
_NORMAL_PATTERNS = [
    "normal chest", "no acute cardiopulmonary", "lungs are clear",
    "no acute disease", "unremarkable", "no acute process",
    "no evidence of malignancy or infection",
]

def is_normal(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in _NORMAL_PATTERNS)


# ── 6. DataFrame pipeline ──────────────────────────────────────────────────────
def label_dataframe(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """
    Add one binary/uncertain column per disease to df.
    Works whether `text_col` contains strings or lists of strings.
    Returns a new DataFrame with the original columns plus disease labels.
    """
    label_records = df[text_col].apply(combine_report_list).tolist()
    labels_df = pd.DataFrame(label_records, index=df.index)
    return pd.concat([df, labels_df], axis=1)


# ── 7. Quick smoke-test on the provided samples ────────────────────────────────
if __name__ == "__main__":
    SAMPLES = [
        # Sample 1 – from document 1 (bilateral effusions, pneumonia uncertain)
        [
            "Findings:  Impression: 1.  There continued to be bilateral airspace opacities which could represent pneumonia or atelectasis.  There are bilateral pleural effusions.  The contours of the effusions suggest that the fluid may be somewhat loculated, particularly on the right.   There is no evidence of pulmonary edema.  Overall cardiac and mediastinal contours are unchanged.  A portion of a biliary stent is again visualized.  Overall, there does not appear to be any significant interval change.  No pneumothorax.",
            "Findings: Heart is upper limits of normal in size, and pulmonary vascularity is normal.  Bibasilar areas of consolidation are present, and may reflect aspiration or infectious pneumonia in the appropriate clinical setting.  Small right and small-to-moderate left pleural effusions are also present.  Biliary stents are noted in the upper abdomen. Impression: ",
        ],
        # Sample 2 – atelectasis, no pneumonia
        [
            "Findings: Lung volumes are low.  Heart size is normal. Linear opacities in the lung bases are compatible with subsegmental atelectasis.  No pleural effusion or pneumothorax is seen. Impression: Subsegmental atelectasis within the lung bases.",
            "Findings: Low lung volumes and atelectasis is seen, and no consolidation, pleural effusion or pulmonary edema is seen.  The cardiac silhouette is enlarged secondary to AP radiograph technique and low lung volumes. Impression: Low lung volumes and atelectasis.  No consolidation to suggest pneumonia.",
        ],
        # Sample 3 – normal
        [
            "Findings: PA and lateral chest radiographs were obtained.  The lungs are well inflated and clear.  No focal consolidation, effusion, or pneumothorax is present. Impression: No acute cardiopulmonary process.",
            "Findings: The lungs are well inflated bilaterally. There are no areas of focal consolidation, masses, lesions, pleural effusion or pneumothorax. Impression: No evidence of malignancy or infection.",
        ],
    ]

    for i, sample in enumerate(SAMPLES, 1):
        labels = combine_report_list(sample)
        print(f"\n{'='*60}")
        print(f"SAMPLE {i}")
        print(f"{'='*60}")
        for d, v in labels.items():
            status = {1: "POSITIVE ✓", -1: "UNCERTAIN ?", 0: "ABSENT  ✗"}[v]
            print(f"  {d:<16} {status}")

    # ── Full CSV run (uncomment when running on Kaggle) ────────────────────────
    train_df = pd.read_csv("/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_train.csv")
    val_df   = pd.read_csv("/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_validate.csv")
    df = pd.concat([train_df, val_df], ignore_index=True)
    df_final = label_dataframe(df, text_col="text")
    df_final.to_csv("mimic_cxr_labeled.csv", index=False)
    print(df_final[list(diseases.keys())].value_counts().head(20))


SAMPLE 1
  pneumonia        UNCERTAIN ?
  edema            ABSENT  ✗
  effusion         POSITIVE ✓
  pneumothorax     ABSENT  ✗
  cardiomegaly     ABSENT  ✗
  atelectasis      UNCERTAIN ?
  consolidation    UNCERTAIN ?
  opacity          ABSENT  ✗

SAMPLE 2
  pneumonia        ABSENT  ✗
  edema            ABSENT  ✗
  effusion         ABSENT  ✗
  pneumothorax     ABSENT  ✗
  cardiomegaly     POSITIVE ✓
  atelectasis      POSITIVE ✓
  consolidation    ABSENT  ✗
  opacity          ABSENT  ✗

SAMPLE 3
  pneumonia        ABSENT  ✗
  edema            ABSENT  ✗
  effusion         ABSENT  ✗
  pneumothorax     ABSENT  ✗
  cardiomegaly     ABSENT  ✗
  atelectasis      ABSENT  ✗
  consolidation    ABSENT  ✗
  opacity          ABSENT  ✗
pneumonia  edema  effusion  pneumothorax  cardiomegaly  atelectasis  consolidation  opacity
 0         0      0         0             0              0           0               0         28622
                                                         1           0  

In [2]:
"""
CMKG-Reasoner Pipeline
======================
Extracts entities & events from MIMIC-CXR multimodal metadata,
constructs a Multimodal Causal Knowledge Graph (CMKG), and serializes
it as a node/edge JSON ready for downstream GNN or LLM reasoning.
 
Input : mimic_cxr_full.csv  (or labeled variant)
Output: cmkg_graph.json, cmkg_triples.csv
"""

'\nCMKG-Reasoner Pipeline\n======================\nExtracts entities & events from MIMIC-CXR multimodal metadata,\nconstructs a Multimodal Causal Knowledge Graph (CMKG), and serializes\nit as a node/edge JSON ready for downstream GNN or LLM reasoning.\n \nInput : mimic_cxr_full.csv  (or labeled variant)\nOutput: cmkg_graph.json, cmkg_triples.csv\n'

In [3]:
import re
import json
import ast
import pandas as pd
from collections import defaultdict

In [4]:
# ─────────────────────────────────────────────────────────────
# 1.  ONTOLOGY  –  entities, events, and causal rules
# ─────────────────────────────────────────────────────────────
 
# Entity types we extract from radiology text
ANATOMICAL_ENTITIES = [
    "left lung", "right lung", "lung", "heart", "cardiac silhouette",
    "diaphragm", "left hemidiaphragm", "right hemidiaphragm",
    "pleura", "mediastinum", "aorta", "trachea", "carina",
    "costophrenic angle", "lung base", "lung apex",
    "right lobe", "left lobe", "hilum", "interstitium",
]
 
PATHOLOGICAL_ENTITIES = [
    "pneumonia", "edema", "effusion", "pleural effusion",
    "pneumothorax", "cardiomegaly", "atelectasis", "consolidation",
    "opacity", "infiltrate", "fibrosis", "calcification",
    "mass", "nodule", "lesion", "fracture", "compression",
    "congestion", "vascular congestion", "interstitial abnormality",
]
 
DEVICE_ENTITIES = [
    "endotracheal tube", "et tube", "nasogastric tube", "ng tube",
    "central line", "pacemaker", "stent", "catheter",
    "right ij", "internal jugular", "drain",
]
 
# Imaging event terms
EVENT_PATTERNS = {
    "worsening":  r"\b(worse|worsening|increased|interval increase|new|developing|developed)\b",
    "improving":  r"\b(improved|improving|decreased|interval decrease|resolving|resolved|no longer)\b",
    "stable":     r"\b(stable|unchanged|similar|no change|no significant change)\b",
    "new_finding":r"\b(new|newly|interval development|developed since)\b",
    "absent":     r"\b(no |absent|not seen|free of|no evidence|resolved|clear)\b",
    "present":    r"\b(present|seen|identified|noted|demonstrated|visualized|consistent with)\b",
}

In [5]:
# ─────────────────────────────────────────────────────────────
# 2.  CAUSAL RULE LIBRARY
#     Format: (cause_entity, effect_entity, relation_label, confidence)
# ─────────────────────────────────────────────────────────────
CAUSAL_RULES = [
    # Cardiac → Pulmonary
    ("cardiomegaly",           "edema",            "causes",            0.85),
    ("cardiomegaly",           "pleural effusion",  "causes",            0.80),
    ("cardiomegaly",           "vascular congestion","causes",           0.82),
    ("vascular congestion",    "edema",             "leads_to",          0.88),
    ("heart failure",          "edema",             "causes",            0.92),
    ("heart failure",          "pleural effusion",  "causes",            0.90),
 
    # Structural → Atelectasis
    ("left hemidiaphragm elevation","atelectasis",  "causes",            0.90),
    ("right hemidiaphragm elevation","atelectasis", "causes",            0.88),
    ("pleural effusion",       "atelectasis",       "causes",            0.85),
    ("low lung volume",        "atelectasis",       "predisposes_to",    0.80),
 
    # Infection / Inflammation
    ("pneumonia",              "consolidation",     "manifests_as",      0.93),
    ("pneumonia",              "opacity",           "manifests_as",      0.88),
    ("aspiration",             "pneumonia",         "causes",            0.87),
    ("infection",              "consolidation",     "causes",            0.82),
 
    # Device / Iatrogenic
    ("endotracheal tube",      "atelectasis",       "risk_factor_for",   0.70),
    ("nasogastric tube",       "aspiration",        "risk_factor_for",   0.65),
    ("central line",           "pneumothorax",      "risk_factor_for",   0.60),
 
    # Structural / Degenerative
    ("kyphosis",               "low lung volume",   "causes",            0.78),
    ("compression fracture",   "kyphosis",          "leads_to",          0.80),
    ("calcification",          "atherosclerosis",   "indicates",         0.88),
 
    # Temporal / Progression
    ("atelectasis",            "pneumonia",         "predisposes_to",    0.65),
    ("edema",                  "opacity",           "manifests_as",      0.80),
    ("pleural effusion",       "opacity",           "manifests_as",      0.82),
]

In [6]:
# ─────────────────────────────────────────────────────────────
# 3.  EXTRACTION FUNCTIONS
# ─────────────────────────────────────────────────────────────
 
def extract_entities(text: str) -> dict:
    """Return dict of {entity_type: [matched_strings]} found in text."""
    t = text.lower()
    found = {"anatomical": [], "pathological": [], "device": []}
    for ent in ANATOMICAL_ENTITIES:
        if ent in t:
            found["anatomical"].append(ent)
    for ent in PATHOLOGICAL_ENTITIES:
        if ent in t:
            found["pathological"].append(ent)
    for ent in DEVICE_ENTITIES:
        if ent in t:
            found["device"].append(ent)
    # Deduplicate
    for k in found:
        found[k] = list(dict.fromkeys(found[k]))
    return found
 
 
def extract_events(text: str, entities: dict) -> list:
    """
    Return list of event dicts:
      {event_type, entity, sentence, modality}
    by matching event patterns near entity mentions.
    """
    events = []
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    all_entities = (entities["anatomical"] +
                    entities["pathological"] +
                    entities["device"])
 
    for sent in sentences:
        s_low = sent.lower()
        for event_type, pattern in EVENT_PATTERNS.items():
            if not re.search(pattern, s_low):
                continue
            for ent in all_entities:
                if ent in s_low:
                    events.append({
                        "event_type": event_type,
                        "entity":     ent,
                        "sentence":   sent.strip(),
                        "modality":   "radiology_text",
                    })
    return events
 
 
def label_entities_from_columns(row: pd.Series) -> list:
    """
    Use the pre-computed binary label columns to assert
    pathological entity nodes with confidence = 1.0.
    """
    disease_cols = [
        "pneumonia", "edema", "effusion", "pneumothorax",
        "cardiomegaly", "atelectasis", "consolidation", "opacity",
    ]
    confirmed = []
    for col in disease_cols:
        if col in row.index:
            val = row[col]
            if val == 1:
                confirmed.append({"entity": col, "status": "present",  "source": "label"})
            elif val == -1:
                confirmed.append({"entity": col, "status": "uncertain","source": "label"})
    return confirmed

In [7]:
# ─────────────────────────────────────────────────────────────
# 4.  CAUSAL GRAPH BUILDER
# ─────────────────────────────────────────────────────────────
 
class CMKGraph:
    """Lightweight in-memory causal knowledge graph."""
 
    def __init__(self):
        self.nodes = {}   # node_id -> {id, label, type, attributes}
        self.edges = []   # {src, dst, relation, confidence, evidence}
        self._nid  = 0
 
    def _get_or_create(self, label: str, node_type: str, attrs: dict = None) -> str:
        key = f"{node_type}:{label}"
        if key not in self.nodes:
            nid = f"n{self._nid:04d}"
            self._nid += 1
            self.nodes[key] = {
                "id": nid, "label": label,
                "type": node_type, "attributes": attrs or {},
            }
        return self.nodes[key]["id"]
 
    def add_entity(self, label: str, etype: str, attrs: dict = None) -> str:
        return self._get_or_create(label, etype, attrs)
 
    def add_event(self, entity_label: str, event_type: str,
                  sentence: str, modality: str) -> str:
        ev_label = f"{event_type}({entity_label})"
        nid = self._get_or_create(ev_label, "event",
                                  {"sentence": sentence, "modality": modality})
        # Link event → entity
        ent_nid = self._get_or_create(entity_label, "entity")
        self.edges.append({
            "src": nid, "dst": ent_nid,
            "relation": "event_of", "confidence": 1.0,
            "evidence": sentence[:120],
        })
        return nid
 
    def apply_causal_rules(self, present_entities: set):
        """Instantiate causal edges for entities present in this study."""
        for (cause, effect, rel, conf) in CAUSAL_RULES:
            # Both endpoints must be present (or the cause is device/structural)
            cause_present = any(cause in e for e in present_entities)
            effect_present = any(effect in e for e in present_entities)
            if cause_present and effect_present:
                src = self._get_or_create(cause,  "entity")
                dst = self._get_or_create(effect, "entity")
                self.edges.append({
                    "src": src, "dst": dst,
                    "relation": rel, "confidence": conf,
                    "evidence": "causal_rule",
                })
 
    def to_dict(self) -> dict:
        return {
            "nodes": list(self.nodes.values()),
            "edges": self.edges,
            "stats": {
                "num_nodes": len(self.nodes),
                "num_edges": len(self.edges),
            },
        }
 
    def to_triples_df(self) -> pd.DataFrame:
        rows = []
        node_map = {v["id"]: v["label"] for v in self.nodes.values()}
        for e in self.edges:
            rows.append({
                "head":     node_map.get(e["src"], e["src"]),
                "relation": e["relation"],
                "tail":     node_map.get(e["dst"], e["dst"]),
                "confidence": e["confidence"],
                "evidence": e.get("evidence", ""),
            })
        return pd.DataFrame(rows)

In [8]:
# ─────────────────────────────────────────────────────────────
# 5.  PER-STUDY PIPELINE
# ─────────────────────────────────────────────────────────────
 
def process_study(row: pd.Series, graph: CMKGraph):
    """Extract entities/events from one row and add to global graph."""
    text = ""
    for col in ["text", "text_augment"]:
        if col in row.index and isinstance(row[col], str):
            text += " " + row[col]
 
    # Handle list-of-strings stored as string repr
    if text.strip().startswith("["):
        try:
            parts = ast.literal_eval(text.strip())
            text = " ".join(p for p in parts if isinstance(p, str))
        except Exception:
            pass
 
    if not text.strip():
        return
 
    subject_id = str(row.get("subject_id", "unknown"))
 
    # 5a. Entity extraction from text
    entities = extract_entities(text)
    for etype, ents in entities.items():
        for ent in ents:
            graph.add_entity(ent, etype, {"subject_id": subject_id})
 
    # 5b. Event extraction
    events = extract_events(text, entities)
    for ev in events:
        graph.add_event(ev["entity"], ev["event_type"],
                        ev["sentence"], ev["modality"])
 
    # 5c. Label-confirmed entities
    confirmed = label_entities_from_columns(row)
    for item in confirmed:
        graph.add_entity(item["entity"], "pathological",
                         {"status": item["status"],
                          "source": item["source"],
                          "subject_id": subject_id})
 
    # 5d. Image modality node (one per study session)
    study_dir = ""
    for col in ["image", "AP", "PA", "Lateral"]:
        if col in row.index and isinstance(row[col], str) and row[col].strip():
            study_dir = row[col].strip()[:80]
            break
    if study_dir:
        img_nid = graph.add_entity(f"img:{subject_id}", "image",
                                   {"path": study_dir})
 
    # 5e. Causal rules
    present = set()
    for etype, ents in entities.items():
        present.update(ents)
    for item in confirmed:
        if item["status"] == "present":
            present.add(item["entity"])
    graph.apply_causal_rules(present)

In [9]:
# ─────────────────────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────────────────────
 
def build_cmkg(csv_path: str,
               out_json: str = "cmkg_graph.json",
               out_triples: str = "cmkg_triples.csv",
               max_rows: int = None) -> CMKGraph:
 
    df = pd.read_csv(csv_path)
    if max_rows:
        df = df.head(max_rows)
 
    graph = CMKGraph()
    for _, row in df.iterrows():
        process_study(row, graph)
 
    g_dict = graph.to_dict()
    with open(out_json, "w") as f:
        json.dump(g_dict, f, indent=2)
 
    triples_df = graph.to_triples_df()
    triples_df.to_csv(out_triples, index=False)
 
    print(f"[CMKG] Nodes : {g_dict['stats']['num_nodes']}")
    print(f"[CMKG] Edges : {g_dict['stats']['num_edges']}")
    print(f"[CMKG] Saved : {out_json}  |  {out_triples}")
 
    # Quick summary of causal edges
    causal = triples_df[triples_df["relation"] != "event_of"]
    print(f"\nTop causal triples (by confidence):")
    print(causal.sort_values("confidence", ascending=False)
                .drop_duplicates(subset=["head","relation","tail"])
                .head(20).to_string(index=False))
 
    return graph
 
 
# ─────────────────────────────────────────────────────────────
# 7.  DEMO  (runs on synthetic data when CSV not present)
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import os
 
    CSV = "/kaggle/working/mimic_cxr_labeled.csv"   # change to your path if needed
 
    if os.path.exists(CSV):
        graph = build_cmkg(CSV, max_rows=500)
    else:
        # ── Synthetic demo on the sample rows visible in the screenshot ──
        print("CSV not found — running synthetic demo on sample data.\n")
 
        sample_rows = [
            {
                "subject_id": 10000032, "text":
                "Findings: There is no acute cardiopulmonary process.",
                "pneumonia": 0, "edema": 0, "effusion": 0,
                "pneumothorax": 0, "cardiomegaly": 0,
                "atelectasis": 0, "consolidation": 0, "opacity": 0,
            },
            {
                "subject_id": 10000764, "text":
                "Findings: PA and lateral chest views. Mild cardiomegaly. "
                "Pulmonary vascular congestion noted. Small pleural effusion.",
                "pneumonia": 0, "edema": 0, "effusion": 1,
                "pneumothorax": 0, "cardiomegaly": 1,
                "atelectasis": 1, "consolidation": 0, "opacity": 0,
            },
            {
                "subject_id": 10000898, "text":
                "Findings: Low lung volumes and atelectasis is seen. "
                "No consolidation. No pleural effusion or pulmonary edema.",
                "pneumonia": 0, "edema": 0, "effusion": 0,
                "pneumothorax": 0, "cardiomegaly": 0,
                "atelectasis": 1, "consolidation": 0, "opacity": 0,
            },
            {
                "subject_id": 10000935, "text":
                "Findings: Lung volumes are chronically very low. "
                "There is more vascularity and bronchial cuffing. "
                "Heart size is borderline enlarged. "
                "Difficult to distinguish between interstitial lung disease "
                "and early congestive heart failure.",
                "pneumonia": 1, "edema": 1, "effusion": 1,
                "pneumothorax": 0, "cardiomegaly": 1,
                "atelectasis": 1, "consolidation": 0, "opacity": 1,
            },
            {
                "subject_id": 10000980, "text":
                "Findings: Endotracheal tube in place. "
                "Left basilar atelectasis and pleural effusion. "
                "No pulmonary edema. Cardiomegaly unchanged.",
                "pneumonia": 1, "edema": 1, "effusion": 1,
                "pneumothorax": 0, "cardiomegaly": 1,
                "atelectasis": 1, "consolidation": 1, "opacity": -1,
            },
        ]
 
        df_demo = pd.DataFrame(sample_rows)
        graph = CMKGraph()
        for _, row in df_demo.iterrows():
            process_study(row, graph)
 
        g_dict = graph.to_dict()
        triples = graph.to_triples_df()
 
        with open("cmkg_graph.json", "w") as f:
            json.dump(g_dict, f, indent=2)
        triples.to_csv("cmkg_triples.csv", index=False)
 
        print(f"Nodes : {g_dict['stats']['num_nodes']}")
        print(f"Edges : {g_dict['stats']['num_edges']}")
 
        print("\n── All triples ──")
        print(triples.to_string(index=False))

[CMKG] Nodes : 871
[CMKG] Edges : 23863
[CMKG] Saved : cmkg_graph.json  |  cmkg_triples.csv

Top causal triples (by confidence):
               head        relation                tail  confidence    evidence
          pneumonia    manifests_as       consolidation        0.93 causal_rule
          pneumonia    manifests_as             opacity        0.88 causal_rule
vascular congestion        leads_to               edema        0.88 causal_rule
   pleural effusion          causes         atelectasis        0.85 causal_rule
       cardiomegaly          causes               edema        0.85 causal_rule
   pleural effusion    manifests_as             opacity        0.82 causal_rule
       cardiomegaly          causes vascular congestion        0.82 causal_rule
              edema    manifests_as             opacity        0.80 causal_rule
       cardiomegaly          causes    pleural effusion        0.80 causal_rule
  endotracheal tube risk_factor_for         atelectasis        0.70 cau